# Exercise 1: exploring a DataFrame

Before dealing with messy files, you explore data that is already clean: the daily weather of 2023 at 13
official KMI stations.

Did you not generate the data yet? Run `uv run python generate_data.py` from the root of the repository.

Run the cell below: click it and press **Shift+Enter**. The first time, VS Code asks which kernel to use:
choose **Select Another Kernel… → Python Environments… →
academy-python-data-analysis**. It should show the pandas version.

In [1]:
import pandas as pd

pd.__version__

'3.1.0'

## A first look

**1.1** Read the file `../../data/raw/kmi_daily_2023.csv` with `pd.read_csv`. Look at the first rows. How many
rows are there? The columns are:

| column | meaning |
| ------ | ------- |
| `name` | the KMI station |
| `timestamp` | the day |
| `temp_min`, `temp_avg`, `temp_max` | lowest, mean and highest temperature of the day (°C) |
| `precip_quantity` | rain (mm) |
| `sun_duration` | sunshine (minutes) |
| `wind_gusts_speed` | strongest gust (m/s) |

Docs: [`pd.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

In [2]:
daily = pd.read_csv("../../data/raw/kmi_daily_2023.csv")
daily.head()

,name,timestamp,temp_min,temp_avg,temp_max,precip_quantity,sun_duration,wind_gusts_speed
0,BEITEM,2023-01-01,10.69,12.11,14.43,2.67,18.83,16.30
1,BEITEM,2023-01-02,3.62,8.54,13.16,0.80,86.58,12.68
2,BEITEM,2023-01-03,2.45,7.53,9.96,1.99,279.75,15.36
3,BEITEM,2023-01-04,9.99,11.67,13.27,2.19,0.00,15.71
4,BEITEM,2023-01-05,9.60,10.79,12.32,1.70,43.67,9.78


In [3]:
len(daily)

4745

4 745 rows: 13 stations, 365 days each.

**1.2** What were the lowest and the highest temperature measured in 2023, at any station? Are there values
missing in any column?

In [4]:
daily.describe()

,temp_min,temp_avg,temp_max,precip_quantity,sun_duration,wind_gusts_speed
count,4743.000000,4743.000000,4743.000000,4745.000000,4745.000000,4743.000000
mean,7.699325,11.554057,15.282288,2.937562,307.113444,10.630525
std,5.551038,6.205628,7.344834,5.491055,280.542252,3.880267
min,-10.350000,-6.140000,-4.530000,0.000000,0.000000,2.520000
25%,3.845000,7.070000,9.745000,0.000000,32.000000,7.740000
50%,8.070000,11.100000,14.500000,0.250000,235.670000,10.070000
75%,11.800000,16.855000,21.350000,3.420000,535.420000,12.980000
max,21.940000,25.720000,33.110000,64.840000,949.670000,31.060000


In [5]:
daily.isna().sum()

name                0
timestamp           0
temp_min            2
temp_avg            2
temp_max            2
precip_quantity     0
sun_duration        0
wind_gusts_speed    2
dtype: int64

The lowest is −10.35 °C (`temp_min`), the highest 33.11 °C (`temp_max`). In `describe`, `count` is 4 743
instead of 4 745 for the temperatures and the wind; `isna().sum()` shows the same: 2 missing values in each
of those columns.

**1.3** Look at the rows with missing values.

Docs: [`any`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.any.html) (optional, not in the slides)

There are several ways. One column at a time:

In [6]:
daily[daily["temp_avg"].isna()]

,name,timestamp,temp_min,temp_avg,temp_max,precip_quantity,sun_duration,wind_gusts_speed
1621,SINT-KATELIJNE-WAVER,2023-06-11,NaN,NaN,NaN,0.0,742.17,7.92
1622,SINT-KATELIJNE-WAVER,2023-06-12,NaN,NaN,NaN,0.0,841.67,8.03


In [7]:
daily[daily["wind_gusts_speed"].isna()]

,name,timestamp,temp_min,temp_avg,temp_max,precip_quantity,sun_duration,wind_gusts_speed
2113,UCCLE,2023-10-16,3.20,7.69,12.13,0.0,469.58,NaN
2114,UCCLE,2023-10-17,7.09,9.09,13.15,0.0,375.33,NaN


Both at once, with `|` for or:

In [8]:
daily[daily["temp_avg"].isna() | daily["wind_gusts_speed"].isna()]

,name,timestamp,temp_min,temp_avg,temp_max,precip_quantity,sun_duration,wind_gusts_speed
1621,SINT-KATELIJNE-WAVER,2023-06-11,NaN,NaN,NaN,0.0,742.17,7.92
1622,SINT-KATELIJNE-WAVER,2023-06-12,NaN,NaN,NaN,0.0,841.67,8.03
2113,UCCLE,2023-10-16,3.20,7.69,12.13,0.0,469.58,NaN
2114,UCCLE,2023-10-17,7.09,9.09,13.15,0.0,375.33,NaN


Or every row with a missing value in any column, without naming the columns:

In [9]:
daily[daily.isna().any(axis="columns")]

,name,timestamp,temp_min,temp_avg,temp_max,precip_quantity,sun_duration,wind_gusts_speed
1621,SINT-KATELIJNE-WAVER,2023-06-11,NaN,NaN,NaN,0.0,742.17,7.92
1622,SINT-KATELIJNE-WAVER,2023-06-12,NaN,NaN,NaN,0.0,841.67,8.03
2113,UCCLE,2023-10-16,3.20,7.69,12.13,0.0,469.58,NaN
2114,UCCLE,2023-10-17,7.09,9.09,13.15,0.0,375.33,NaN


Four different days: the temperatures are missing at Sint-Katelijne-Waver on 11 and 12 June, the wind at
Uccle on 16 and 17 October. So looking at `temp_avg` alone shows only half of them.

## Bonus

Explore the data we'll be using today: the files in `../../data/raw/`.

There is no single answer. `weather_export.csv` does not read cleanly with `pd.read_csv` alone: that comes
in part 4.